<div dir="rtl" align="right">

# گرادیان بوستینگ برای رگرسیون

درسی از مدرسه پایتون و هوش مصنوعی، دکتر بهروز طاهری

## ایده در یک نگاه
جنگل تصادفی درخت‌ها را **موازی و مستقل** می‌سازد. بوستینگ درخت‌ها را **پشت سر هم** می‌سازد: هر درخت جدید یاد می‌گیرد **خطای باقی‌مانده** درخت‌های قبلی را درست کند. مثل دانش‌آموزی که بعد از هر امتحان فقط روی اشتباه‌هایش کار می‌کند.

## الگوریتم
- شروع: $F_0(x)=\bar y$ (میانگین).
- برای $m=1,\dots,M$:
- باقی‌مانده را حساب کن: $r_i=y_i-F_{m-1}(x_i)$.
- یک درخت کوچک $h_m$ روی $(x_i, r_i)$ آموزش بده.
- به‌روز کن: $F_m(x)=F_{m-1}(x)+\eta\,h_m(x)$.

$\eta$ **نرخ یادگیری** است (مثلاً 0.1): هر درخت فقط بخش کوچکی از اصلاح را انجام می‌دهد. چرا «گرادیان»؟ چون باقی‌مانده دقیقاً منفی گرادیان خطای مربعی است؛ پس بوستینگ همان گرادیان کاهشی در فضای توابع است.

## مثال عددی
$y=[3, 5, 10]$، $\eta=0.5$:

| مرحله | پیش‌بینی | باقی‌مانده |
|---|---|---|
| $F_0$ | [6, 6, 6] | [−3, −1, 4] |
| درخت ۱ یاد می‌گیرد | [−2, −2, 4] | |
| $F_1=F_0+0.5h_1$ | [5, 5, 8] | [−2, 0, 2] |
| درخت ۲ ... | | خطا کمتر و کمتر |

> [بعد از ۱ و ۲۰ درخت، و خطای آموزش و آزمون بر حسب تعداد درخت. خطای آزمون از جایی به بعد زیاد می‌شود.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/gbr_stages.png)

*بعد از ۱ و ۲۰ درخت، و خطای آموزش و آزمون بر حسب تعداد درخت. خطای آزمون از جایی به بعد زیاد می‌شود.*

[[lab:reg_gbr]]

## هایپرپارامترها و توازنشان
| هایپرپارامتر | معمول | نکته |
|---|---|---|
| `n_estimators` | 100 تا 1000 | برخلاف جنگل، زیاد بودنش بیش‌برازش می‌دهد |
| `learning_rate` | 0.01 تا 0.1 | کمتر = درخت بیشتر لازم، معمولاً دقیق‌تر |
| `max_depth` | 2 تا 5 | درخت‌های **کوچک** |
| `subsample` | 0.5 تا 1 | کمتر از ۱ = بوستینگ تصادفی، مقاوم‌تر |
| `n_iter_no_change` | مثلاً 10 | **توقف زودهنگام** با داده اعتبارسنجی |

> قاعده طلایی: learning_rate را کم کن و n_estimators را با توقف زودهنگام پیدا کن.



</div>

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor

# Gradient boosting written by hand, on a noisy sine
rng = np.random.default_rng(0)
x = rng.uniform(0, 1, 80)[:, None]; y = np.sin(2 * np.pi * x.ravel()) + rng.normal(0, 0.2, 80)
xt = rng.uniform(0, 1, 300)[:, None]; yt = np.sin(2 * np.pi * xt.ravel()) + rng.normal(0, 0.2, 300)

lr, F, Ft = 0.1, np.full(80, y.mean()), np.full(300, y.mean())
for m in range(1, 301):
    tree = DecisionTreeRegressor(max_depth=2).fit(x, y - F)   # learn the residuals
    F += lr * tree.predict(x); Ft += lr * tree.predict(xt)
    if m in (1, 10, 50, 100, 300):
        print(f"trees={m:3d}  train MSE={np.mean((y - F) ** 2):.3f}  test MSE={np.mean((yt - Ft) ** 2):.3f}")

<div dir="rtl" align="right">



## نسخه‌های سریع‌تر
`HistGradientBoostingRegressor` ویژگی‌ها را دسته‌بندی (histogram) می‌کند و روی داده بزرگ بسیار سریع است؛ داده گمشده را هم مستقیم می‌پذیرد. کتابخانه‌های XGBoost، LightGBM و CatBoost هم همین ایده را دارند و در مسابقات داده‌های جدولی بسیار موفق‌اند (این‌ها را در Colab اجرا کن).



</div>

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_score

X, y = load_diabetes(return_X_y=True)
gb = GradientBoostingRegressor(n_estimators=2000, learning_rate=0.02, max_depth=2, subsample=0.8,
                               validation_fraction=0.2, n_iter_no_change=20, random_state=0).fit(X, y)
print("Trees chosen by early stopping:", gb.n_estimators_)
for name, m in [("GradientBoosting", GradientBoostingRegressor(n_estimators=gb.n_estimators_, learning_rate=0.02, max_depth=2, subsample=0.8, random_state=0)),
                ("HistGradientBoosting", HistGradientBoostingRegressor(learning_rate=0.05, max_depth=3, random_state=0))]:
    print(name, round(cross_val_score(m, X, y, cv=5, scoring="r2").mean(), 3))

<div dir="rtl" align="right">



| جنگل تصادفی | گرادیان بوستینگ |
|---|---|
| درخت‌های عمیق و مستقل | درخت‌های کوچک و پشت سر هم |
| واریانس را کم می‌کند | بایاس را کم می‌کند |
| تنظیم کم | تنظیم دقیق‌تر لازم |
| بیش‌برازش سخت | با درخت زیاد بیش‌برازش می‌کند |

</div>

<div dir="rtl" align="right">

### تمرین

در کد اول learning_rate را 1.0 و 0.01 بگذار. کدام سریع‌تر بیش‌برازش می‌کند؟

</div>

In [ ]:
# Your answer here